In [1]:
# -*- coding: utf-8 -*-
"""marco-bustamante_tarea-02.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/mbustamc/cc66t_laboratorios/blob/master/tarea_02/marco-bustamante_tarea-02.ipynb

# Carga de librerías
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import random
import os

from torchvision import transforms
from sklearn.preprocessing import normalize
from PIL import Image
from tqdm import tqdm
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")



Usando dispositivo: cpu


In [27]:
def load_dino_v2(dino_v2):
    #dino_v2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
    models = {}
    for key, value in dino_v2.items():
        models[key] = torch.hub.load('facebookresearch/dinov2', value)
        models[key] = models[key].to(device)
        models[key].eval()
        print(f"¡Modelo {value} cargado exitosamente!")
        print(f"Dimensión de salida del modelo: {models[key].embed_dim}")
    return models


dino_v2 = {
    'dino_v2_s':'dinov2_vits14',
    'dino_v2_b': 'dinov2_vitb14',
    'dino_v2_l':'dinov2_vitl14',
    #'dino_v2_g':'dinov2_vitg14'
    }

dino_models_v2 = load_dino_v2(dino_v2)


Using cache found in /home/mbustamc/.cache/torch/hub/facebookresearch_dinov2_main


¡Modelo dinov2_vits14 cargado exitosamente!
Dimensión de salida del modelo: 384


Using cache found in /home/mbustamc/.cache/torch/hub/facebookresearch_dinov2_main


¡Modelo dinov2_vitb14 cargado exitosamente!
Dimensión de salida del modelo: 768


Using cache found in /home/mbustamc/.cache/torch/hub/facebookresearch_dinov2_main


¡Modelo dinov2_vitl14 cargado exitosamente!
Dimensión de salida del modelo: 1024


In [29]:
#!gdown --id 1-LqJ2bS_T8XOx5TClDq31MCkPPoXO4bZ
#!unzip dinov3-main.zip



def load_dino_v3(dino_v3):
    models = {}
    for model in dino_v3:
        #!gdown --id {model['descarga']}
        model_name = model['nombre']
        models[model_name] = torch.hub.load(repo_or_dir='./dinov3-main', model=model_name, source='local', weights=model['pesos'])
        models[model_name] = models[model_name].to(device)
        models[model_name].eval()
        print(f"¡Modelo {model_name} cargado exitosamente!")
        print(f"Dimensión de salida del modelo: {models[model_name].embed_dim}")
    return models


dinov3_vits16plus = {
    'nombre':'dinov3_vits16plus',
    'descarga':'1fH2rq53x6JY6zE_WBKH5vP_ZGq4jvj46',
    'pesos':'dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth'
    }

dinov3_vitb16 = {
    'nombre':'dinov3_vitb16',
    'descarga':'1Fznrc_pDwp7iaUBhAoWUKPI1vsGpHy8m',
    'pesos':'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'
    }


dinov3_vitl16 = {
    'nombre':'dinov3_vitl16',
    'descarga':'1CfyML_a7PkVpt4Lhy-3yNxSfUgsFfgCs',
    'pesos':'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
    }


dino_v3 = [dinov3_vits16plus, dinov3_vitb16,]

dino_models_v3 = load_dino_v3(dino_v3)

dino_models = dino_models_v2 | dino_models_v3

print(f'Keys: {dino_models.keys()}')

¡Modelo dinov3_vits16plus cargado exitosamente!
Dimensión de salida del modelo: 384
¡Modelo dinov3_vitb16 cargado exitosamente!
Dimensión de salida del modelo: 768
Keys: dict_keys(['dino_v2_s', 'dino_v2_b', 'dino_v2_l', 'dinov3_vits16plus', 'dinov3_vitb16'])


In [30]:

# Descargar el archivo zip desde Google Drive
!gdown --id 11-TD6add7zZaukIB8l2cVnBTJgsTjo8n

# Descomprimir el archivo
!unzip dataset_ecom_mini.zip

# Cargar los metadatos del conjunto de datos
data_dir = Path('eval')
df = pd.read_csv('eval.csv', delimiter=';')

print(f"El conjunto de datos contiene {len(df)} imágenes")
print(f"\nDistribución de categorías:")
print(df['GlobalCategoryEN'].value_counts())


/home/mbustamc/Documentos/diplomados/dia_2025/cc66t_reconocimiento-visual/cc66t_laboratorios/.venv/lib/python3.11/site-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=11-TD6add7zZaukIB8l2cVnBTJgsTjo8n
To: /home/mbustamc/Documentos/diplomados/dia_2025/cc66t_reconocimiento-visual/cc66t_laboratorios/tarea_02/dataset_ecom_mini.zip
100%|██████████████████████████████████████| 10.6M/10.6M [00:00<00:00, 28.0MB/s]
Archive:  dataset_ecom_mini.zip
   creating: eval/
  inflating: eval/45elec.jpg         
  inflating: eval/im.qf53qh.input.jpg  
  inflating: eval/8toys.jpg          
  inflating: eval/im.rf7hhh.input.jpg  
  inflating: eval/im.rys7rv.input.jpg  
  inflating: eval/226pets.jpg        
  inflating: eval/0off.jpg           
  inflating: eval/im.7wd4b3.input.jpg  
  inflating: eval/443pets.jpg    

In [38]:
def transform(image):
    transform_pipeline = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transform_pipeline(image)

def encode_image(image_path, model):
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model(img_tensor)

    return features.cpu().numpy().flatten()


def generar_embeddings(model, df, data_dir, nombre_modelo):

    embeddings = []
    valid_names = []

    print(f"\n Generando embeddings con {nombre_modelo}...")
    model.eval()
    model.to(device)

    for filename in tqdm(df['Title'].values):
        img_path = data_dir / f"{filename}.jpg"
        if img_path.exists():
            try:
                img = Image.open(img_path).convert('RGB')
                img_tensor = transform(img).unsqueeze(0).to(device)
                with torch.no_grad():
                    feat = model(img_tensor)
                embeddings.append(feat.cpu().numpy().flatten())
                valid_names.append(filename)
            except Exception as e:
                print(f"Error con {filename}: {e}")

    embeddings = np.array(embeddings)
    print(f"{len(embeddings)} imágenes procesadas con {nombre_modelo}")
    return embeddings, valid_names


In [39]:
# Función auxiliar para calcular similitud coseno
def cosine_similarity(vec1, vec2):
    """
    vec1, vec2: (numpy arrays)
    Puntuación de similitud entre 0 y 1 (1 = idéntico)
    """
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)



def calcular_similitud(embeddings):
    """
    embeddings: matriz N x D
    Retorna la matriz N x N de similitud coseno.
    """
    embeddings_normalized = normalize(embeddings, axis=1)
    similarity_matrix = np.dot(embeddings_normalized, embeddings_normalized.T)
    return similarity_matrix



def display_image_comparison(img_paths, titles, similarities=None):
    """
    Muestra imágenes lado a lado con sus títulos y puntuaciones de similitud.
    """
    n_images = len(img_paths)
    fig, axes = plt.subplots(1, n_images, figsize=(5*n_images, 5))

    if n_images == 1:
        axes = [axes]

    for idx, (img_path, title) in enumerate(zip(img_paths, titles)):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')

        if similarities is not None and idx > 0:
            axes[idx].set_title(f"{title}\nSimilitud: {similarities[idx-1]:.4f}", fontsize=12)
        else:
            axes[idx].set_title(title, fontsize=12)

    plt.tight_layout()
    plt.show()

def procesar_varios_modelos(modelos_dict, df, data_dir):
    """
    modelos_dict: {nombre_modelo: modelo_torch}
    df: dataframe con columna 'Title'
    """
    resultados = {}

    for nombre, modelo in modelos_dict.items():
        emb, valid_names = generar_embeddings(modelo, df, data_dir, nombre)
        sim = calcular_similitud(emb)
        resultados[nombre] = {
            'embeddings': emb,
            'valid_names': valid_names,
            'similarity_matrix': sim
        }
        print(f"💾 {nombre}: embeddings {emb.shape}, matriz {sim.shape}")

    return resultados


In [43]:
# Si quieres procesar solo modelos DINOv3

resultados_dino = procesar_varios_modelos(dino_models, df, data_dir)


 Generando embeddings con dino_v2_s...


100%|██████████| 300/300 [00:49<00:00,  6.03it/s]


300 imágenes procesadas con dino_v2_s
💾 dino_v2_s: embeddings (300, 384), matriz (300, 300)

 Generando embeddings con dino_v2_b...


100%|██████████| 300/300 [02:32<00:00,  1.96it/s]


300 imágenes procesadas con dino_v2_b
💾 dino_v2_b: embeddings (300, 768), matriz (300, 300)

 Generando embeddings con dino_v2_l...


...

300 imágenes procesadas con dino_v2_l
💾 dino_v2_l: embeddings (300, 1024), matriz (300, 300)

 Generando embeddings con dinov3_vits16plus...


100%|██████████| 300/300 [00:51<00:00,  5.81it/s]


300 imágenes procesadas con dinov3_vits16plus
💾 dinov3_vits16plus: embeddings (300, 384), matriz (300, 300)

 Generando embeddings con dinov3_vitb16...


100%|██████████| 300/300 [02:14<00:00,  2.23it/s]

300 imágenes procesadas con dinov3_vitb16
💾 dinov3_vitb16: embeddings (300, 768), matriz (300, 300)


In [45]:
for model_name, data in resultados_dino.items():
    print(f"\nAnálisis de similitud para el modelo: {model_name}")
    similarity_matrix = data['similarity_matrix']       
    print(f"Forma de la matriz de similitud: {similarity_matrix.shape}")
    diag_vals = similarity_matrix.diagonal()[:5]
    print(f"Valores diagonales (auto-similitud) deberían ser ~1.0: {diag_vals}")
    print(f"\nEstadísticas de similitud:")
    print(f"  Mín:   {similarity_matrix.min():.4f}")
    print(f"  Máx:   {similarity_matrix.max():.4f}")
    print(f"  Media: {similarity_matrix.mean():.4f}")
    print(f"  Desv.: {similarity_matrix.std():.4f}")
    print("=" * 60)




Análisis de similitud para el modelo: dino_v2_s
Forma de la matriz de similitud: (300, 300)
Valores diagonales (auto-similitud) deberían ser ~1.0: [1.        0.9999999 0.9999999 0.9999999 1.       ]

Estadísticas de similitud:
  Mín:   -0.1963
  Máx:   1.0000
  Media: 0.0607
  Desv.: 0.1130

Análisis de similitud para el modelo: dino_v2_b
Forma de la matriz de similitud: (300, 300)
Valores diagonales (auto-similitud) deberían ser ~1.0: [1.0000002  0.99999976 1.0000001  0.9999999  0.9999999 ]

Estadísticas de similitud:
  Mín:   -0.1597
  Máx:   1.0000
  Media: 0.0480
  Desv.: 0.0973

Análisis de similitud para el modelo: dino_v2_l
Forma de la matriz de similitud: (300, 300)
Valores diagonales (auto-similitud) deberían ser ~1.0: [1.0000001  1.         0.9999996  0.99999976 1.0000001 ]

Estadísticas de similitud:
  Mín:   -0.1045
  Máx:   1.0000
  Media: 0.0477
  Desv.: 0.0944

Análisis de similitud para el modelo: dinov3_vits16plus
Forma de la matriz de similitud: (300, 300)
Valores di

In [ ]:

def find_and_display_top_k_similar(similarity_matrix, query_idx, valid_filenames, data_dir, df, k=10, model_name="modelo"):
    """
    Encuentra y muestra las K imágenes más similares a una imagen de consulta
    usando una matriz de similitud coseno.

    Args:
        similarity_matrix (np.ndarray): Matriz NxN de similitudes coseno.
        query_idx (int): Índice de la imagen de consulta.
        valid_filenames (list[str]): Lista de nombres de archivo (sin extensión).
        data_dir (Path): Carpeta donde están las imágenes.
        df (pd.DataFrame): DataFrame con metadatos (debe contener columna 'Title' y 'GlobalCategoryEN').
        k (int): Número de imágenes similares a mostrar.
        model_name (str): Nombre del modelo usado para calcular similitud.

    Returns:
        dict: Diccionario con índices y similitudes del top K.
    """
    # --- Verificación de entrada ---
    n = similarity_matrix.shape[0]
    if query_idx >= n:
        raise IndexError(f"query_idx={query_idx} fuera de rango (N={n})")

    # --- Calcular similitudes ---
    similarities = similarity_matrix[query_idx]
    top_indices = np.argsort(similarities)[::-1]
    top_indices = [i for i in top_indices if i != query_idx][:k]
    top_similarities = similarities[top_indices]

    # --- Datos de la imagen de consulta ---
    query_filename = valid_filenames[query_idx]
    query_path = data_dir / f"{query_filename}.jpg"
    query_category = df[df["Title"] == query_filename]["GlobalCategoryEN"].values[0]

    print(f"\n=== Modelo: {model_name} ===")
    print(f"Imagen de Consulta: {query_filename} (Categoría: {query_category})")
    print(f"Top {k} imágenes más similares:\n" + "-" * 80)

    for i, (idx, sim) in enumerate(zip(top_indices, top_similarities)):
        fname = valid_filenames[idx]
        cat = df[df["Title"] == fname]["GlobalCategoryEN"].values[0]
        print(f"{i+1:2d}. {fname:25s} | Categoría: {cat:20s} | Similitud: {sim:.4f}")

    # --- Mostrar imágenes ---
    n_cols = 5
    n_rows = (k + n_cols - 1) // n_cols + 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 3*n_rows))
    axes = axes.flatten()

    # Imagen de consulta
    query_img = Image.open(query_path)
    axes[0].imshow(query_img)
    axes[0].set_title(f"CONSULTA\n{query_filename}\n{query_category}", fontsize=10, fontweight='bold', color='red')
    axes[0].axis("off")

    for i in range(1, n_cols):
        axes[i].axis("off")

    # Mostrar top K
    for i, (idx, sim) in enumerate(zip(top_indices, top_similarities)):
        fname = valid_filenames[idx]
        img_path = data_dir / f"{fname}.jpg"
        category = df[df["Title"] == fname]["GlobalCategoryEN"].values[0]

        img = Image.open(img_path)
        ax_idx = n_cols + i
        axes[ax_idx].imshow(img)
        axes[ax_idx].set_title(f"#{i+1}: {fname}\n{category}\nSim: {sim:.4f}", fontsize=9)
        axes[ax_idx].axis("off")

    # Ocultar subplots no usados
    for i in range(n_cols + k, len(axes)):
        axes[i].axis("off")

    plt.suptitle(f"Modelo: {model_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    return {
        "query_idx": query_idx,
        "top_indices": top_indices,
        "similarities": top_similarities.tolist()
    }

In [ ]:
def compute_average_precision(query_idx, similarity_matrix, valid_filenames, df, k=None):
    """
    Calcula la Precisión Promedio (AP) para una sola consulta.

    Args:
        query_idx: índice de la imagen de consulta.
        similarity_matrix: matriz N x N de similitudes.
        valid_filenames: lista de nombres válidos (en el mismo orden que similarity_matrix).
        df: DataFrame original con 'Title' y 'GlobalCategoryEN'.
        k: número de resultados top a considerar (None = todos los resultados).

    Returns:
        average_precision (float), num_relevant (int)
    """
    # --- Categoría de la consulta ---
    query_filename = valid_filenames[query_idx]
    query_category = df.loc[df['Title'] == query_filename, 'GlobalCategoryEN'].values[0]

    # --- Categorías de todas las imágenes ---
    categories = [
        df.loc[df['Title'] == fn, 'GlobalCategoryEN'].values[0]
        for fn in valid_filenames
    ]

    # --- Máscara de elementos relevantes ---
    relevant_mask = np.array([cat == query_category for cat in categories])
    relevant_mask[query_idx] = False  # excluir la misma imagen
    num_relevant = relevant_mask.sum()

    if num_relevant == 0:
        return 0.0, 0

    # --- Ordenar por similitud (descendente) ---
    similarities = similarity_matrix[query_idx]
    ranked_indices = np.argsort(similarities)[::-1]

    # Eliminar el índice de la consulta
    ranked_indices = ranked_indices[ranked_indices != query_idx]

    # Limitar a top-K si se solicita
    if k is not None:
        ranked_indices = ranked_indices[:k]

    # --- Calcular AP ---
    precision_sum = 0.0
    num_relevant_seen = 0

    for i, idx in enumerate(ranked_indices):
        if relevant_mask[idx]:
            num_relevant_seen += 1
            precision_at_i = num_relevant_seen / (i + 1)
            precision_sum += precision_at_i

    average_precision = precision_sum / num_relevant
    return average_precision, num_relevant


# --- 2️⃣ Calcular Mean Average Precision (MAP) ---
def compute_MAP(similarity_matrix, valid_filenames, df, k=None):
    """
    Calcula la Media de Precisión Promedio (MAP) sobre todas las imágenes de consulta.

    Args:
        similarity_matrix: matriz N x N de similitudes.
        valid_filenames: lista de nombres válidos.
        df: DataFrame con columnas 'Title' y 'GlobalCategoryEN'.
        k: número de resultados top a considerar (None = todos los resultados).

    Returns:
        mean_ap: precisión promedio global
        per_image_ap: lista con el AP individual de cada imagen
    """
    aps = []
    total_relevant = 0

    for i in range(len(valid_filenames)):
        ap, n_rel = compute_average_precision(i, similarity_matrix, valid_filenames, df, k)
        aps.append(ap)
        total_relevant += n_rel

    mean_ap = np.mean(aps)
    print(f"📊 MAP@{k if k else 'all'} = {mean_ap:.4f}  (promedio sobre {len(valid_filenames)} consultas)")
    return mean_ap, aps


# --- 3️⃣ Ejemplo de uso con tus resultados ---
# Supongamos que ya tienes tus embeddings y similitud para cada modelo:
# resultados = procesar_varios_modelos(modelos_dict, df, data_dir)

for nombre, res in resultados.items():
    sim = res['similarity_matrix']
    valid_names = res['valid_names']
    mean_ap, _ = compute_MAP(sim, valid_names, df, k=10)
    print(f"✅ MAP@10 para {nombre}: {mean_ap:.4f}")